# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Use CUDA async allocator to reduce fragmentation:
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# Suppress TensorFlow logging (0: ALL, 1: INFO, 2: WARNING, 3: ERROR):
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# # If it fails to determine best cudnn convolution algorithm
# os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [ ]:
# # Disable all auto-JIT clustering at the process level
# os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 1000
EPOCHS = 3

SAMPLER_SEED = 111
TRAIN_SEED = 111

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
# tf.config.experimental.enable_op_determinism() #! Spektral does not support this yet

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
JIT_COMPILE = False  #! Spektral does not support this yet

In [ ]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "best_val_accuracy"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "maximize"

In [ ]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [ ]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [ ]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels(s008_path="./data/s008", s009_path="./data/s009")

In [ ]:
(
    x_lidar_train,
    x_lidar_val,
    x_lidar_test,
    x_coord_train,
    x_coord_val,
    x_coord_test,
    y_train,
    y_val,
    y_test,
) = load_dataset_raw_sparse_labels(
    s008_coord_csv="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s008/raw_data/CoordVehiclesRxPerScene_s008.csv",
    s009_coord_csv="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s009/raw_data/CoordVehiclesRxPerScene_s009.csv",
    s008_lidar_folder="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s008/processed_raw_data/lidar_data_s008",
    s009_lidar_folder="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s009/processed_raw_data/lidar_data_s009",
    s008_beam_output_path="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s008/baseline_data/beam_output/beams_output_s008.npz",
    s009_beam_output_path="/media/matheus/SSD-2/matheus/datasets/RayWise/Raymobtime_s009/baseline_data/beam_output/beams_output_test.npz",
    data_seed=1000,
    # stratify=True,
    stratify_by_class=True,
    remove_null_labels=True,
)

## Hyperparameters

In [ ]:
kparams = KParams.default()
kparams.learning_rate = 7e-5

## 5. Model Definition

In [ ]:
def build_model(trial: optuna.Trial, kparams: dict, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— CNN ——————————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 4)

    for i in range(num_conv_layers):
        x = build_cnn1d(
            trial=trial,
            kparams=kparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(3, 9),
            kernel_size_step=2,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        #! pool size = 1 means no downsampling
        pool_size = trial.suggest_int(f"pool_size_{i}", 1, 4, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)

    cnn_out = x  # keep 3D shape here

    # ---------------------- Dense branch decisions --------------------------- #
    use_cnn_as_dense = trial.suggest_categorical("use_cnn_as_dense", [True, False])
    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 3)
    pooling_type = trial.suggest_categorical("pooling_type", ["flatten", "max", "average"])

    if use_cnn_as_dense:
        x = cnn_out  # still 3D
        for i in range(num_dense_layers):
            x = build_dense_as_conv1d(
                trial=trial,
                kparams=kparams,
                x=x,
                name_prefix=f"cnn_dense_{i}",
                filters_range=(250, 600),
                filters_step=50,
                kernel_initializer=initializer,
            )
        # collapse after the conv-as-dense stack
        if pooling_type == "flatten":
            x = layers.Flatten(name="flatten_after_cnn_dense")(x)
        elif pooling_type == "max":
            x = layers.GlobalMaxPooling1D(name="global_max_pooling")(x)
        else:
            x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(x)
    else:
        # collapse first
        if pooling_type == "flatten":
            x = layers.Flatten(name="flatten_cnn_output")(cnn_out)
        elif pooling_type == "max":
            x = layers.GlobalMaxPooling1D(name="global_max_pooling")(cnn_out)
        else:
            x = layers.GlobalAveragePooling1D(name="global_avg_pooling")(cnn_out)

        for i in range(num_dense_layers):
            x = build_dnn(
                trial=trial,
                kparams=kparams,
                x=x,
                name_prefix=f"dense_{i}",
                units_range=(250, 600),
                units_step=50,
                dropout_rate_range=(0.0, 0.4),
                dropout_rate_step=0.2,
                kernel_initializer=initializer,
            )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        162,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=kparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=JIT_COMPILE,
        steps_per_execution=32,
    )

    return model

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    *,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    # Set Python, NumPy, Keras and TensorFlow seeds
    set_random_seed(TRAIN_SEED)

    global x_lidar_train, x_lidar_val, x_lidar_test
    global x_coord_train, x_coord_val, x_coord_test
    global y_train, y_val, y_test

    global s009_coord_input, s009_lidar_input, s009_y
    global s008_coord_input, s008_lidar_input, s008_y_train

    backup_dir = kwargs["backup_dir"]
    model_dir = kwargs["model_dir"]
    fig_dir = kwargs["fig_dir"]
    tensorboard_dir = kwargs["tensorboard_dir"]
    logs_dir = kwargs["logs_dir"]
    history_dir = kwargs["history_dir"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)
        s008_coord_input = coord_scaler.transform(s008_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, kparams=kparams, show_summary=False)
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                # "model_size": 80,  # Maximum model size in MB
                "memory_mb": 8000,  # Maximum memory training usage in MB
                # "param": 1e6,  # Maximum number of parameters
                # "flops": 1e9,  # Maximum number of FLOPs
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
                reduce_lr_patience=None,
                pruning_interval=10,
            ),
            verbose=2,
        )

        trial.set_user_attr("best_train_accuracy", float(max(history.history.get("accuracy", []))))
        trial.set_user_attr("best_val_accuracy", float(max(history.history.get("val_accuracy", []))))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values) if DIRECTION == "minimize" else max(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            test_runs=10,
            device="gpu/0",
            stats_to_measure=(
                "parameters",
                "model_size",
                "flops",
                "macs",
                "summary",
                "inference_latency",
                # "cpu_util_percent",
                # "cpu_power_rapl_w",
                # "ram_used_bytes",
                # "ram_util_percent",
                # "gpu_util_percent",
                # "gpu_mem_used_bytes",
                # "gpu_power_w",
            ),
            extra_attrs=None,
            verbose=1,
        )

        # ——————————————————————————— Evaluate on full s009 ——————————————————————————— #
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ————————————————————————————— Evaluate on s008 ————————————————————————————— #
        test_loss_s008, test_acc_s008 = model.evaluate(
            [s008_lidar_input, s008_coord_input], s008_y_train, batch_size=BATCH_SIZE, verbose=0
        )
        trial.set_user_attr("test_accuracy_s008_full", float(test_acc_s008))
        trial.set_user_attr("test_loss_s008_full", float(test_loss_s008))

        # ————————————————————————— Evaluate on the test set ————————————————————————— #
        test_loss_s009, test_acc_s009 = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=BATCH_SIZE, verbose=0
        )
        trial.set_user_attr("test_accuracy_test_set", float(test_acc_s009))
        trial.set_user_attr("test_loss_test_set", float(test_loss_s009))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss #history.history["val_accuracy"]  # Value to minimize or maximize
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
        )

## Main

In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    epochs=EPOCHS,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    extra_attrs=[
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009_full",
        "test_loss_s009_full",
        "test_accuracy_s008_full",
        "test_loss_s008_full",
        "test_accuracy_test_set",
        "test_loss_test_set",
    ],
    variance_threshold=None,
    prune_threshold=None,
    patience=None,
)